# Airbnb Pricing Analysis — European Cities
## Machine Learning: Price Prediction with XGBoost

In this notebook, we build a regression model to predict Airbnb nightly prices
across 10 European cities using the XGBoost algorithm.

**Dataset:** `airbnb_clean.csv` — 51,571 listings · 10 cities · 21 columns

**Model:** XGBoost Regressor — predicts `price_per_night` from listing features

> 🗒️ **Note:** Run `01_Data_Cleaning_Airbnb.ipynb` first to generate the `airbnb_clean.csv` file.

## 1. Mount Google Drive

We mount Google Drive to access the dataset files stored in the cloud.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Import Libraries

We import the libraries needed for data preparation, model training and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 3. Load Data

We load the cleaned dataset generated in `01_Data_Cleaning_Airbnb.ipynb`.

In [ ]:
FOLDER = '/content/drive/MyDrive/Portfolio/Project3-Airbnb/Dataset/'

df = pd.read_csv(FOLDER + 'airbnb_clean.csv')
print(f'Shape: {df.shape}')
df.head()

## 4. Feature Engineering

Before training, we prepare the dataset for the model:
- Drop columns that would cause data leakage or are redundant.
- Encode categorical variables as numeric values.
- Separate features (X) from the target variable (y).

### 4.1 Drop Non-Predictive Columns

We remove the following columns:
- `longitude` and `latitude` — would cause overfitting (the model would memorize the map).
- `bedroom_label` — redundant with `bedrooms`.
- `price` — the raw total price; our target is `price_per_night`, which is derived from it.

In [ ]:
df_ml = df.drop(columns=['longitude', 'latitude', 'bedroom_label', 'price'])
print(f'Columns kept for modelling: {df_ml.shape[1]}')
print(df_ml.columns.tolist())

### 4.2 One-Hot Encoding

We convert categorical columns to numeric using One-Hot Encoding.
We use `drop_first=False` because XGBoost handles correlated features natively
and we want the model to have access to all categories.

In [ ]:
cat_cols = ['city', 'day_type', 'room_type']
df_ml = pd.get_dummies(df_ml, columns=cat_cols, drop_first=False)
print(f'Columns after encoding: {df_ml.shape[1]}')
print(df_ml.columns.tolist())

### 4.3 Define Features and Target

We separate the feature matrix `X` from the target variable `y`.
The model will learn to predict `price_per_night` from all remaining columns.

In [ ]:
X = df_ml.drop(columns=['price_per_night'])
y = df_ml['price_per_night']

print(f'Features (X): {X.shape[1]} columns, {X.shape[0]:,} rows')
print(f'Target  (y): {y.shape[0]:,} values')

### 4.4 Train / Test Split

We split the data into a training set (80%) and a test set (20%).
The model learns from the training set and is evaluated on the test set — data it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]:,} rows')
print(f'Test set     : {X_test.shape[0]:,} rows')
print(f'Features     : {X_train.shape[1]} columns')

## 5. Model Training

We use **XGBoost** (Extreme Gradient Boosting), a tree-based ensemble algorithm
that builds models sequentially, with each tree correcting the errors of the previous one.
It is well-suited for tabular data and handles mixed feature types naturally.

### 5.1 Instantiate the Model

We configure the main hyperparameters:
- `n_estimators` — number of trees to build.
- `learning_rate` — how much each tree contributes to the final prediction.
- `max_depth` — maximum depth of each tree; limits complexity and prevents overfitting.
- `n_jobs=-1` — uses all available CPU cores for faster training.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print('Model ready.')

### 5.2 Train the Model

We fit the model on the training set.
XGBoost will build 300 decision trees sequentially to minimise prediction error.

In [ ]:
model.fit(X_train, y_train)
print('Training complete.')

## 6. Model Evaluation

We evaluate the model on the test set — the 20% of data it has never seen during training.

### 6.1 Performance Metrics

We use two metrics:
- **MAE (Mean Absolute Error)** — average prediction error in euros. Lower is better.
- **R² (R-squared)** — proportion of price variance explained by the model. Closer to 1 is better.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print('--- Model Results (Test Set) ---')
print(f'MAE : ±{mae:.2f} EUR per night')
print(f'R²  : {r2:.3f}')
print(f'\nThe model explains {r2*100:.1f}% of the variance in listing prices.')

### 6.2 Actual vs Predicted Prices

We plot actual prices against the model's predictions.
A perfect model would place all points on the diagonal line.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.2, color='#2947d3', s=8)
plt.plot([0, 500], [0, 500], color='#1a3a5c', linewidth=1.5, linestyle='--', label='Perfect prediction')
plt.title('Actual vs Predicted Price (EUR/night)')
plt.xlabel('Actual Price (EUR)')
plt.ylabel('Predicted Price (EUR)')
plt.xlim(0, 500)
plt.ylim(0, 500)
plt.legend()
plt.tight_layout()
plt.show()

### 6.3 Feature Importance

We analyze which features have the greatest influence on the model's predictions.
This helps us understand what drives Airbnb pricing beyond simple averages.

In [ ]:
importance = pd.DataFrame({
    'feature'   : X.columns,
    'importance': model.feature_importances_ * 100
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

print('--- Top 10 Most Influential Features ---')
print(importance.head(10).to_string(index=False))

In [ ]:
top10 = importance.head(10).sort_values('importance')

plt.figure(figsize=(9, 5))
plt.barh(top10['feature'], top10['importance'], color='#2947d3')
plt.title('Top 10 Feature Importance (%)')
plt.xlabel('Importance (%)')
plt.tight_layout()
plt.show()

## 7. Predictions for Power BI

We apply the trained model to the full dataset to generate predictions for every listing.
These predictions are then added as new columns and exported to a CSV for use in Power BI.

### 7.1 Predict on Full Dataset

We run the model on `X` (all rows, not just the test set) to obtain a predicted price
for every listing in the dataset.

In [ ]:
all_predictions = model.predict(X)

print(f'Predictions generated: {len(all_predictions):,}')

### 7.2 Add ML Columns to Original Dataset

We attach the predictions to the original `df` (which retains `longitude` and `latitude`
for the Power BI map), and calculate two derived columns:
- `predicted_price_ml` — the model's estimated fair price for each listing.
- `price_deviation` — difference between the actual and predicted price.
  A negative value means the listing is cheaper than expected.

In [ ]:
df_powerbi = df.copy()
df_powerbi['predicted_price_ml'] = pd.Series(all_predictions).round(2).values
df_powerbi['price_deviation']    = (df_powerbi['price_per_night'] - df_powerbi['predicted_price_ml']).round(2)

print(df_powerbi[['price_per_night', 'predicted_price_ml', 'price_deviation']].describe().round(2))

### 7.3 Investment Opportunity Label

We create a categorical label based on the price deviation:
- **Underpriced (Bargain)** — listing costs more than €15 below the predicted fair price.
- **Overpriced** — listing costs more than €15 above the predicted fair price.
- **Fair Price** — within ±€15 of the prediction.

This column feeds the investment opportunity chart in Power BI.

In [ ]:
def label_opportunity(deviation):
    if deviation < -15: return 'Underpriced (Bargain)'
    elif deviation > 15: return 'Overpriced'
    else: return 'Fair Price'

df_powerbi['investment_opportunity'] = df_powerbi['price_deviation'].apply(label_opportunity)

print('Investment opportunity distribution:')
print(df_powerbi['investment_opportunity'].value_counts())

### 7.4 Export to CSV

We save the enriched dataset to Google Drive for use in Power BI.

In [ ]:
output_path = FOLDER + 'airbnb_ml_powerbi.csv'
df_powerbi.to_csv(output_path, index=False)

print(f'File saved: airbnb_ml_powerbi.csv')
print(f'Final dimensions: {df_powerbi.shape[0]:,} rows x {df_powerbi.shape[1]} columns')

In [ ]:
# Final validation
df_check = pd.read_csv(output_path, nrows=5)
print(f'Columns: {df_check.columns.tolist()}')
print(f'Shape confirmed: {df_check.shape}')

## 8. Conclusions

### Model Performance

**Results**
- The XGBoost model achieves an R² of 0.737, meaning it explains 73.7% of the variance in Airbnb nightly prices.
- The Mean Absolute Error is ±15.20 EUR per night — a reasonable margin given the natural variability of short-term rental pricing.
- Performance is consistent across cities, with no single market driving the results.

**Feature Importance**
- City is the strongest predictor of price, confirming that location is the dominant factor in Airbnb pricing.
- Person capacity and room type are the next most influential features.
- Host-related features (superhost status, multiple listings) have relatively low predictive power.

**Investment Opportunities**
- Listings labelled *Underpriced (Bargain)* have actual prices more than €15 below the model's fair value estimate.
- These represent potential opportunities for guests seeking value, or for hosts to adjust their pricing strategy.
- Cities with lower average prices (Athens, Budapest) tend to have a higher proportion of underpriced listings.

**Model Limitations**
- The model does not have access to listing photos, text descriptions, or seasonal demand — all of which influence real-world pricing.
- Predictions are based on structural features only and should be interpreted as a pricing baseline, not an exact forecast.